# Demo end-to-end: happy path de flows en Pylondrina

En esta demo quiero probar el camino feliz completo del bloque Trip -> Flow:
importar viajes desde un CSV fuente, validarlos formalmente, construir flujos OD agregados y exportarlos a un layout interoperable para visualización.

La idea aquí no es tensionar el módulo con casos patológicos, sino mostrar un caso rico y bien configurado, donde todo sale bien y queda trazabilidad suficiente para inspeccionar el proceso.

### Configuración inicial

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "synthetic"

### 0. Setup e imports

In [2]:
from pathlib import Path
import json
import shutil

import pandas as pd

RAW_CSV_PATH = DATA_PATH / "demo_happy_export.csv"
ARTIFACT_ROOT = DATA_PATH / "demo_outputs"
EXPORT_ROOT = ARTIFACT_ROOT / "flow_exports"

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

print("REPO_ROOT:", REPO_ROOT)
print("RAW_CSV_PATH:", RAW_CSV_PATH)
print("EXPORT_ROOT:", EXPORT_ROOT)

REPO_ROOT: D:\Memoria\repos\pylondrina
RAW_CSV_PATH: D:\Memoria\repos\pylondrina\data\synthetic\demo_happy_export.csv
EXPORT_ROOT: D:\Memoria\repos\pylondrina\data\synthetic\demo_outputs\flow_exports


## 0. Setup e imports

Voy a trabajar con la API pública real del módulo y con un pequeño helper para visualizar issues de forma cómoda.

In [3]:
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema
from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.validation import ValidationOptions, validate_trips
from pylondrina.transforms.flows import FlowBuildOptions, build_flows
from pylondrina.export.flows import ExportFlowsOptions, export_flows


def issues_to_df(issues):
    if not issues:
        return pd.DataFrame(columns=["level", "code", "message", "field", "source_field", "row_count", "details"])
    rows = []
    for issue in issues:
        rows.append(
            {
                "level": issue.level,
                "code": issue.code,
                "message": issue.message,
                "field": issue.field,
                "source_field": issue.source_field,
                "row_count": issue.row_count,
                "details": issue.details,
            }
        )
    return pd.DataFrame(rows)


def make_field(name, dtype, *, required=False, constraints=None, domain=None):
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )

## 1. Carga inicial del CSV crudo

Primero quiero revisar rápidamente el dataset fuente antes de importarlo. Me interesa confirmar que:
- tiene un tamaño suficientemente grande,
- trae nombres de campos "fuente",
- y todavía no trae H3, porque en esta demo quiero que el import los derive a resolución 12.

In [4]:
raw_df = pd.read_csv(RAW_CSV_PATH)

print("Shape del CSV fuente:", raw_df.shape)
print("Cantidad de columnas:", len(raw_df.columns))
print("¿Trae H3 de origen?", "origin_h3_index" in raw_df.columns)
print("¿Trae H3 de destino?", "destination_h3_index" in raw_df.columns)

display(raw_df.head(5))
display(raw_df[["modo_fuente", "sexo_fuente", "tipo_dia_fuente"]].head(10))
display(raw_df[["modo_fuente"]].value_counts().rename("n").reset_index().head(10))
display(raw_df[["sexo_fuente"]].value_counts().rename("n").reset_index())
display(raw_df[["tipo_dia_fuente"]].value_counts().rename("n").reset_index())

Shape del CSV fuente: (18000, 40)
Cantidad de columnas: 40
¿Trae H3 de origen? False
¿Trae H3 de destino? False


,id_movimiento,id_usuario,id_viaje,orden_movimiento,lon_origen,lat_origen,lon_destino,lat_destino,ts_origen,ts_destino,comuna_origen,comuna_destino,offset_tz_min,hora_origen_local,hora_destino_local,peso_viaje,secuencia_modos,modo_fuente,proposito_fuente,tipo_dia_fuente,franja_horaria_fuente,sexo_fuente,tramo_edad_fuente,quintil_ingreso_fuente,activity_status,education_level,travel_time_bucket,season,fare_payment_type,home_tenure,travel_time_min,fare_amount,public_transport_route_code,distance_route_m,household_income_clp,occupation_type,job_sector,work_schedule,origin_stop_id,destination_stop_id
0,m00000,u2155,tm_00000,0,-70.722291,-33.416870,-70.769244,-33.387348,2026-03-08T03:26:00Z,2026-03-08T04:59:00Z,Ñuñoa,Quilicura,-180,03:26,04:59,2.927,metro+walk,metro,work,holiday,night,male,0-14,unknown,studying,university,21-40,summer,free_transfer,owned,93.0,760.0,L4,5212.0,1567303.0,informal,education,shift,STOP_0143,STOP_1011
1,m00001,u3432,tm_00001,0,-70.613602,-33.604783,-70.573264,-33.595883,2026-03-01T09:26:00Z,2026-03-01T09:32:00Z,Recoleta,Providencia,-180,09:26,09:32,2.267,car+walk+bus,other,leisure,Fin de Semana,midday,other,0-14,3,homemaker,none,60+,normal_period,free_transfer,rented,6.0,1520.0,401,7413.9,1502697.0,public_sector,services,shift,STOP_4871,STOP_0966
2,m00002,u3432,tm_00001,1,-70.577250,-33.442752,-70.593671,-33.440256,2026-03-04T09:28:00Z,2026-03-04T09:35:00Z,Independencia,Recoleta,-180,09:28,09:35,4.626,train,metro,work,weekend,midday,Hombre,0-14,2,studying,none,21-40,summer,integrated_fare,other,7.0,760.0,401,5417.7,1289973.0,self_employed,commerce,night_shift,STOP_3221,STOP_2323
3,m00003,u1852,tm_00002,0,-70.800601,-33.357680,-70.799504,-33.317213,2026-03-05T11:05:00Z,2026-03-05T12:57:00Z,La Florida,Maipú,-180,11:05,12:57,3.841,metro+train+train,motorcycle,health,DOMINGO,midday,Mujer,45-54,unknown,studying,none,60+,vacation,not_applicable,other,112.0,1200.0,210,917.9,882061.0,employee,health,shift,STOP_1176,STOP_2228
4,m00004,u1852,tm_00002,1,-70.761925,-33.339011,-70.721093,-33.337207,2026-03-02T18:03:00Z,2026-03-02T19:44:00Z,Santiago,Las Condes,-180,18:03,19:44,1.561,walk+car+walk,metro,shopping,weekday,morning,female,35-44,1,studying,university,60+,normal_period,card,rented,101.0,1520.0,L1,2331.3,877938.0,public_sector,commerce,part_time,STOP_0529,STOP_3122


,modo_fuente,sexo_fuente,tipo_dia_fuente
0,metro,male,holiday
1,other,other,Fin de Semana
2,metro,Hombre,weekend
3,motorcycle,Mujer,DOMINGO
4,metro,female,weekday
5,other,unknown,Laboral
6,motorcycle,Hombre,Fin de Semana
7,car,Hombre,Laboral
8,motorcycle,unknown,weekday
9,bicycle,unknown,Laboral


,modo_fuente,n
0,bus,1679
1,walk,1673
2,scooter,1672
3,motorcycle,1662
4,taxi,1653
5,train,1641
6,other,1633
7,ride_hailing,1626
8,car,1608
9,metro,1590


,sexo_fuente,n
0,other,3114
1,male,3000
2,Mujer,2990
3,Hombre,2967
4,unknown,2967
5,female,2962


,tipo_dia_fuente,n
0,Laboral,10366
1,Fin de Semana,2529
2,weekend,1289
3,DOMINGO,1281
4,holiday,1268
5,weekday,1267


## 2. Definición del contrato de importación

Aquí fijo el contrato que quiero aplicar en el import. Me interesa que el schema sea lo bastante rico como para validar bien el dataset, pero sin volver la demo innecesariamente pesada.

También dejo explícitas las correspondencias:
- de campos (nombres fuente -> nombres Golondrina),
- y de valores (etiquetas fuente -> valores canónicos),
para que el import haga una normalización real antes de la validación.

In [5]:
SOURCE_FIELD_CORRESPONDENCE = {
    "movement_id": "id_movimiento",
    "user_id": "id_usuario",
    "origin_longitude": "lon_origen",
    "origin_latitude": "lat_origen",
    "destination_longitude": "lon_destino",
    "destination_latitude": "lat_destino",
    "origin_time_utc": "ts_origen",
    "destination_time_utc": "ts_destino",
    "trip_id": "id_viaje",
    "movement_seq": "orden_movimiento",
    "origin_municipality": "comuna_origen",
    "destination_municipality": "comuna_destino",
    "timezone_offset_min": "offset_tz_min",
    "origin_time_local_hhmm": "hora_origen_local",
    "destination_time_local_hhmm": "hora_destino_local",
    "trip_weight": "peso_viaje",
    "mode_sequence": "secuencia_modos",
    "mode": "modo_fuente",
    "purpose": "proposito_fuente",
    "day_type": "tipo_dia_fuente",
    "time_period": "franja_horaria_fuente",
    "user_gender": "sexo_fuente",
    "user_age_group": "tramo_edad_fuente",
    "income_quintile": "quintil_ingreso_fuente",
}

VALUE_CORRESPONDENCE = {
    "user_gender": {
        "Hombre": "male",
        "Mujer": "female",
    },
    "day_type": {
        "Laboral": "weekday",
        "Fin de Semana": "weekend",
        "DOMINGO": "holiday",
    },
}

MODE_VALUES = [
    "walk", "bicycle", "scooter", "motorcycle", "car",
    "taxi", "ride_hailing", "bus", "metro", "train", "other",
]
PURPOSE_VALUES = [
    "home", "work", "education", "shopping", "errand",
    "health", "leisure", "transfer", "other",
]
DAY_TYPE_VALUES = ["weekday", "weekend", "holiday"]
TIME_PERIOD_VALUES = ["night", "morning", "midday", "afternoon", "evening"]
USER_GENDER_VALUES = ["female", "male", "other", "unknown"]
USER_AGE_GROUP_VALUES = ["0-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65-plus", "unknown"]
INCOME_QUINTILE_VALUES = ["1", "2", "3", "4", "5", "unknown"]

ACTIVITY_STATUS_VALUES = ["working", "studying", "homemaker", "retired", "unemployed", "other"]
EDUCATION_LEVEL_VALUES = ["none", "primary", "secondary", "technical", "university", "postgraduate"]
SEASON_VALUES = ["summer", "winter", "normal_period", "vacation"]
FARE_PAYMENT_TYPE_VALUES = ["cash", "card", "integrated_fare", "free_transfer", "not_applicable"]
HOME_TENURE_VALUES = ["owned", "rented", "loaned", "other"]

trip_fields = {
    # Required
    "movement_id": make_field(
        "movement_id", "string", required=True,
        constraints={"nullable": False, "unique": True},
    ),
    "user_id": make_field(
        "user_id", "string", required=True,
        constraints={"nullable": False},
    ),
    "origin_longitude": make_field(
        "origin_longitude", "float", required=True,
        constraints={"nullable": False, "range": {"min": -180.0, "max": 180.0}},
    ),
    "origin_latitude": make_field(
        "origin_latitude", "float", required=True,
        constraints={"nullable": False, "range": {"min": -90.0, "max": 90.0}},
    ),
    "destination_longitude": make_field(
        "destination_longitude", "float", required=True,
        constraints={"nullable": False, "range": {"min": -180.0, "max": 180.0}},
    ),
    "destination_latitude": make_field(
        "destination_latitude", "float", required=True,
        constraints={"nullable": False, "range": {"min": -90.0, "max": 90.0}},
    ),
    "origin_h3_index": make_field(
        "origin_h3_index", "string", required=True,
        constraints={"nullable": False, "h3": {"require_valid": True, "resolution": 12, "allow_mixed_resolution": False}},
    ),
    "destination_h3_index": make_field(
        "destination_h3_index", "string", required=True,
        constraints={"nullable": False, "h3": {"require_valid": True, "resolution": 12, "allow_mixed_resolution": False}},
    ),
    "origin_time_utc": make_field(
        "origin_time_utc", "datetime", required=True,
        constraints={"nullable": False, "datetime": {}},
    ),
    "destination_time_utc": make_field(
        "destination_time_utc", "datetime", required=True,
        constraints={"nullable": False, "datetime": {}},
    ),
    "trip_id": make_field(
        "trip_id", "string", required=True,
        constraints={"nullable": False},
    ),
    "movement_seq": make_field(
        "movement_seq", "int", required=True,
        constraints={"nullable": False, "range": {"min": 0}},
    ),

    # Base / optional
    "origin_municipality": make_field("origin_municipality", "string"),
    "destination_municipality": make_field("destination_municipality", "string"),
    "timezone_offset_min": make_field("timezone_offset_min", "int", constraints={"range": {"min": -720, "max": 840}}),
    "origin_time_local_hhmm": make_field("origin_time_local_hhmm", "string"),
    "destination_time_local_hhmm": make_field("destination_time_local_hhmm", "string"),
    "trip_weight": make_field("trip_weight", "float", constraints={"range": {"min": 0.0}}),
    "mode_sequence": make_field("mode_sequence", "string"),
    "mode": make_field("mode", "categorical", domain=DomainSpec(values=MODE_VALUES, extendable=True)),
    "purpose": make_field("purpose", "categorical", domain=DomainSpec(values=PURPOSE_VALUES, extendable=True)),
    "day_type": make_field("day_type", "categorical", domain=DomainSpec(values=DAY_TYPE_VALUES, extendable=True)),
    "time_period": make_field("time_period", "categorical", domain=DomainSpec(values=TIME_PERIOD_VALUES, extendable=True)),
    "user_gender": make_field("user_gender", "categorical", domain=DomainSpec(values=USER_GENDER_VALUES, extendable=True)),
    "user_age_group": make_field("user_age_group", "categorical", domain=DomainSpec(values=USER_AGE_GROUP_VALUES, extendable=True)),
    "income_quintile": make_field("income_quintile", "categorical", domain=DomainSpec(values=INCOME_QUINTILE_VALUES, extendable=True)),

    # Extras "ricas" que quiero validar formalmente también
    "travel_time_min": make_field("travel_time_min", "float", constraints={"range": {"min": 0.0}}),
    "fare_amount": make_field("fare_amount", "float", constraints={"range": {"min": 0.0}}),
    "activity_status": make_field("activity_status", "categorical", domain=DomainSpec(values=ACTIVITY_STATUS_VALUES, extendable=True)),
    "education_level": make_field("education_level", "categorical", domain=DomainSpec(values=EDUCATION_LEVEL_VALUES, extendable=True)),
    "season": make_field("season", "categorical", domain=DomainSpec(values=SEASON_VALUES, extendable=True)),
    "fare_payment_type": make_field("fare_payment_type", "categorical", domain=DomainSpec(values=FARE_PAYMENT_TYPE_VALUES, extendable=True)),
    "home_tenure": make_field("home_tenure", "categorical", domain=DomainSpec(values=HOME_TENURE_VALUES, extendable=True)),
}

TRIP_SCHEMA = TripSchema(
    version="1.1",
    fields=trip_fields,
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "origin_time_utc",
        "destination_time_utc",
        "trip_id",
        "movement_seq",
    ],
    semantic_rules=None,
)

print("Campos en el schema:", len(TRIP_SCHEMA.fields))
display(pd.DataFrame([{"field": k, "dtype": v.dtype, "required": v.required} for k, v in TRIP_SCHEMA.fields.items()]).head(30))

Campos en el schema: 33


,field,dtype,required
0,movement_id,string,True
1,user_id,string,True
2,origin_longitude,float,True
3,origin_latitude,float,True
4,destination_longitude,float,True
5,destination_latitude,float,True
6,origin_h3_index,string,True
7,destination_h3_index,string,True
8,origin_time_utc,datetime,True
9,destination_time_utc,datetime,True


## 3. Importación al formato Golondrina

Aquí quiero que el import haga tres cosas bien:
- normalizar nombres de campos desde la convención fuente,
- estandarizar valores categóricos donde corresponde,
- y derivar H3 directamente a resolución 12 desde las coordenadas del CSV.

In [6]:
import_options = ImportOptions(
    keep_extra_fields=True,
    selected_fields=None,
    strict=True,
    strict_domains=True,
    single_stage=False,
    source_timezone=None,
)

trips, import_report = import_trips_from_dataframe(
    raw_df,
    TRIP_SCHEMA,
    source_name="demo_trip_to_flow_happy_path_v1",
    options=import_options,
    field_correspondence=SOURCE_FIELD_CORRESPONDENCE,
    value_correspondence=VALUE_CORRESPONDENCE,
    provenance={
        "source": {
            "name": "demo_trip_to_flow_happy_path_v1",
            "entity": "synthetic_trips_csv",
        },
        "notes": [
            "demo end-to-end de flows",
            "H3 omitido en fuente y derivado en import",
        ],
    },
    h3_resolution=12,
)

assert import_report.ok is True
print("Import OK:", import_report.ok)
display(import_report.summary)
display(issues_to_df(import_report.issues))

Import OK: True


{'rows_in': 18000,
 'rows_out': 18000,
 'n_fields_mapped': 24,
 'n_domain_mappings_applied': 5}

,level,code,message,field,source_field,row_count,details
0,info,IMP.METADATA.DATASET_ID_CREATED,Se generó dataset_id para el dataset importado: 'tripds_0ec20aea36a2479b958805ab2bdc46a0'.,None,None,None,"{'dataset_id': 'tripds_0ec20aea36a2479b958805ab2bdc46a0', 'generator': 'uuid4_hex', 'stored_in': 'metadata.dataset_id'}"


### Exploración del resultado del import

Ahora quiero revisar si el dataset ya quedó realmente "Golondrina-friendly":
- columnas canónicas presentes,
- H3 derivados,
- metadata temporal y espacial razonable,
- y columnas extra preservadas.

In [7]:
print("Shape de trips.data:", trips.data.shape)
print("Cantidad de columnas:", len(trips.data.columns))
print("is_validated después del import:", trips.metadata.get("is_validated"))

display(trips.data.head(5))

cols_to_inspect = [
    "movement_id",
    "trip_id",
    "movement_seq",
    "mode",
    "purpose",
    "day_type",
    "user_gender",
    "trip_weight",
    "origin_h3_index",
    "destination_h3_index",
]
display(trips.data[cols_to_inspect].head(10))

print("metadata['temporal']:")
display(trips.metadata.get("temporal"))

print("metadata['h3']:")
display(trips.metadata.get("h3"))

print("Último evento registrado:")
display(trips.metadata.get("events", [])[-1])

Shape de trips.data: (18000, 42)
Cantidad de columnas: 42
is_validated después del import: False


,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_municipality,destination_municipality,timezone_offset_min,origin_time_local_hhmm,destination_time_local_hhmm,trip_weight,mode_sequence,mode,purpose,day_type,time_period,user_gender,user_age_group,income_quintile,activity_status,education_level,travel_time_bucket,season,fare_payment_type,home_tenure,travel_time_min,fare_amount,public_transport_route_code,distance_route_m,household_income_clp,occupation_type,job_sector,work_schedule,origin_stop_id,destination_stop_id,origin_h3_index,destination_h3_index
0,m00000,u2155,tm_00000,0,-70.722291,-33.416870,-70.769244,-33.387348,2026-03-08 03:26:00+00:00,2026-03-08 04:59:00+00:00,Ñuñoa,Quilicura,-180,03:26,04:59,2.927,metro+walk,metro,work,holiday,night,male,0-14,unknown,studying,university,21-40,summer,free_transfer,owned,93.0,760.0,L4,5212.0,1567303.0,informal,education,shift,STOP_0143,STOP_1011,8cb2c55736317ff,8cb2c550dc9cdff
1,m00001,u3432,tm_00001,0,-70.613602,-33.604783,-70.573264,-33.595883,2026-03-01 09:26:00+00:00,2026-03-01 09:32:00+00:00,Recoleta,Providencia,-180,09:26,09:32,2.267,car+walk+bus,other,leisure,weekend,midday,other,0-14,3,homemaker,none,60+,normal_period,free_transfer,rented,6.0,1520.0,401,7413.9,1502697.0,public_sector,services,shift,STOP_4871,STOP_0966,8cb2c5732cd31ff,8cb2c5730a087ff
2,m00002,u3432,tm_00001,1,-70.577250,-33.442752,-70.593671,-33.440256,2026-03-04 09:28:00+00:00,2026-03-04 09:35:00+00:00,Independencia,Recoleta,-180,09:28,09:35,4.626,train,metro,work,weekend,midday,male,0-14,2,studying,none,21-40,summer,integrated_fare,other,7.0,760.0,401,5417.7,1289973.0,self_employed,commerce,night_shift,STOP_3221,STOP_2323,8cb2c50b44c83ff,8cb2c50b6d44dff
3,m00003,u1852,tm_00002,0,-70.800601,-33.357680,-70.799504,-33.317213,2026-03-05 11:05:00+00:00,2026-03-05 12:57:00+00:00,La Florida,Maipú,-180,11:05,12:57,3.841,metro+train+train,motorcycle,health,holiday,midday,female,45-54,unknown,studying,none,60+,vacation,not_applicable,other,112.0,1200.0,210,917.9,882061.0,employee,health,shift,STOP_1176,STOP_2228,8cb2c55057425ff,8cb2c5520718bff
4,m00004,u1852,tm_00002,1,-70.761925,-33.339011,-70.721093,-33.337207,2026-03-02 18:03:00+00:00,2026-03-02 19:44:00+00:00,Santiago,Las Condes,-180,18:03,19:44,1.561,walk+car+walk,metro,shopping,weekday,morning,female,35-44,1,studying,university,60+,normal_period,card,rented,101.0,1520.0,L1,2331.3,877938.0,public_sector,commerce,part_time,STOP_0529,STOP_3122,8cb2c552c6b05ff,8cb2c552db73bff


,movement_id,trip_id,movement_seq,mode,purpose,day_type,user_gender,trip_weight,origin_h3_index,destination_h3_index
0,m00000,tm_00000,0,metro,work,holiday,male,2.927,8cb2c55736317ff,8cb2c550dc9cdff
1,m00001,tm_00001,0,other,leisure,weekend,other,2.267,8cb2c5732cd31ff,8cb2c5730a087ff
2,m00002,tm_00001,1,metro,work,weekend,male,4.626,8cb2c50b44c83ff,8cb2c50b6d44dff
3,m00003,tm_00002,0,motorcycle,health,holiday,female,3.841,8cb2c55057425ff,8cb2c5520718bff
4,m00004,tm_00002,1,metro,shopping,weekday,female,1.561,8cb2c552c6b05ff,8cb2c552db73bff
5,m00005,tm_00003,0,other,shopping,weekday,unknown,0.768,8cb2c50b579cbff,8cb2c50b2a557ff
6,m00006,tm_00003,1,motorcycle,shopping,weekend,male,4.305,8cb2c50166e91ff,8cb2c5010a9e1ff
7,m00007,tm_00004,0,car,education,weekday,male,1.527,8cb2c518d0e4dff,8cb2c5189034bff
8,m00008,tm_00005,0,motorcycle,education,weekday,unknown,1.421,8cb2c55552c53ff,8cb2c5554673dff
9,m00009,tm_00005,1,bicycle,home,weekday,unknown,2.893,8cb2c552c3b59ff,8cb2c5524cf6dff


metadata['temporal']:


{'tier': 'tier_1',
 'fields_present': ['origin_time_utc',
  'destination_time_utc',
  'origin_time_local_hhmm',
  'destination_time_local_hhmm'],
 'normalization': {'origin_time_utc': {'status': 'string_tzaware_to_utc',
   'tz_kind': 'none',
   'n_nat': 0},
  'destination_time_utc': {'status': 'string_tzaware_to_utc',
   'tz_kind': 'none',
   'n_nat': 0}}}

metadata['h3']:


{'resolution': 12,
 'source_fields': [['origin_latitude', 'origin_longitude'],
  ['destination_latitude', 'destination_longitude']],
 'derived_fields': ['origin_h3_index', 'destination_h3_index']}

Último evento registrado:


{'op': 'import_trips',
 'ts_utc': '2026-04-06T07:12:03.425905+00:00',
 'parameters': {'keep_extra_fields': True,
  'selected_fields': None,
  'strict': True,
  'strict_domains': True,
  'single_stage': False,
  'h3_resolution': 12,
  'source_name': 'demo_trip_to_flow_happy_path_v1',
  'source_timezone': None},
 'summary': {'input_rows': 18000,
  'output_rows': 18000,
  'rows_dropped': 0,
  'n_fields_mapped': 24,
  'n_domain_mappings_applied': 5,
  'columns_added': ['origin_h3_index', 'destination_h3_index'],
  'columns_deleted': [],
  'domains_extended_count': 0,
  'temporal_tier': 'tier_1',
  'temporal_notes': None},
 'issues_summary': {'counts': {}, 'by_code': {}}}

## 4. Validación formal inicial

En esta demo quiero que la validación sea realmente exigente:
- required fields,
- tipos y formatos,
- constraints,
- dominios en modo full,
- consistencia temporal,
- y duplicados por movement_id.

Si todo quedó bien configurado, aquí debería validarse sin problemas.

In [8]:
validation_options = ValidationOptions(
    strict=True,
    max_issues=200,
    sample_rows_per_issue=5,
    validate_required_fields=True,
    validate_types_and_formats=True,
    validate_constraints=True,
    validate_domains="full",
    domains_sample_frac=0.01,
    domains_min_in_domain_ratio=1.0,
    validate_temporal_consistency=True,
    validate_duplicates=True,
    duplicates_subset=("movement_id",),
    allow_partial_od_spatial=False,
)

validation_report = validate_trips(trips, options=validation_options)

assert validation_report.ok is True
assert trips.metadata["is_validated"] is True

print("Validation OK:", validation_report.ok)
display(validation_report.summary)
display(issues_to_df(validation_report.issues))
print("is_validated después de validate:", trips.metadata.get("is_validated"))
print("Último evento registrado:")
display(trips.metadata.get("events", [])[-1])

Validation OK: True


{'ok': True,
 'n_rows': 18000,
 'n_issues': 0,
 'n_errors': 0,
 'n_warnings': 0,
 'n_info': 0,
 'counts_by_level': {'error': 0, 'warning': 0, 'info': 0},
 'counts_by_code': {},
 'checked_fields': ['activity_status',
  'day_type',
  'destination_h3_index',
  'destination_latitude',
  'destination_longitude',
  'destination_municipality',
  'destination_time_local_hhmm',
  'destination_time_utc',
  'education_level',
  'fare_amount',
  'fare_payment_type',
  'home_tenure',
  'income_quintile',
  'mode',
  'mode_sequence',
  'movement_id',
  'movement_seq',
  'origin_h3_index',
  'origin_latitude',
  'origin_longitude',
  'origin_municipality',
  'origin_time_local_hhmm',
  'origin_time_utc',
  'purpose',
  'season',
  'time_period',
  'timezone_offset_min',
  'travel_time_min',
  'trip_id',
  'trip_weight',
  'user_age_group',
  'user_gender',
  'user_id'],
 'checks_executed': {'required_fields': True,
  'types_and_formats': True,
  'constraints': True,
  'domains': True,
  'temporal_con

,level,code,message,field,source_field,row_count,details


is_validated después de validate: True
Último evento registrado:


{'op': 'validate_trips',
 'ts_utc': '2026-04-06T07:14:05Z',
 'parameters': {'strict': True,
  'max_issues': 200,
  'sample_rows_per_issue': 5,
  'validate_required_fields': True,
  'validate_types_and_formats': True,
  'validate_constraints': True,
  'validate_domains': 'full',
  'domains_sample_frac': 0.01,
  'domains_min_in_domain_ratio': 1.0,
  'validate_temporal_consistency': True,
  'validate_duplicates': True,
  'duplicates_subset': ['movement_id'],
  'allow_partial_od_spatial': False},
 'summary': {'ok': True,
  'n_rows': 18000,
  'n_issues': 0,
  'n_errors': 0,
  'n_warnings': 0,
  'n_info': 0,
  'counts_by_level': {'error': 0, 'warning': 0, 'info': 0},
  'counts_by_code': {},
  'checked_fields': ['activity_status',
   'day_type',
   'destination_h3_index',
   'destination_latitude',
   'destination_longitude',
   'destination_municipality',
   'destination_time_local_hhmm',
   'destination_time_utc',
   'education_level',
   'fare_amount',
   'fare_payment_type',
   'home_tenu

## 5. Construcción de flujos OD agregados

Para no dejar la agregación demasiado dispersa, aquí voy a:
- agregar a resolución H3 7,
- segmentar solo por `mode` y `user_gender`,
- agregar temporalmente por día usando el tiempo de origen,
- y exigir un mínimo de viajes por flujo.

Es una configuración moderada: suficientemente rica para mostrar valor, pero sin volver el resultado demasiado fragmentado.

In [44]:
build_options = FlowBuildOptions(
    h3_resolution=5,
    group_by=["mode", "user_gender"],
    time_aggregation="day",
    time_basis="origin",
    min_trips_per_flow=5,
    keep_flow_to_trips=False,
    require_validated=True,
    strict=False,
    max_issues=200,
)

flow_dataset, flow_build_report = build_flows(trips, options=build_options)

assert flow_build_report.ok is True
assert len(flow_dataset.flows) > 0

print("Build OK:", flow_build_report.ok)
display(flow_build_report.summary)
display(issues_to_df(flow_build_report.issues))

Build OK: True


{'n_trips_in': 18000,
 'n_trips_eligible': 18000,
 'n_trips_dropped': 0,
 'n_flows_out': 879,
 'n_flow_to_trips_rows': None}

,level,code,message,field,source_field,row_count,details


### Exploración de los flujos construidos

Aquí me interesa ver:
- cómo quedó el `FlowDataset`,
- qué tan grande es la tabla de flujos,
- cuál fue la configuración efectiva de agregación,
- y si el pipeline dejó trazabilidad suficiente.

In [31]:
print("Shape de flow_dataset.flows:", flow_dataset.flows.shape)
print("is_validated del FlowDataset:", flow_dataset.metadata.get("is_validated"))

display(flow_dataset.flows.head(10))

cols_flow = [
    "flow_id",
    "origin_h3_index",
    "destination_h3_index",
    "mode",
    "user_gender",
    "window_start_utc",
    "window_end_utc",
    "flow_count",
    "flow_value",
]
display(flow_dataset.flows[cols_flow].head(10))

print("aggregation_spec:")
display(flow_dataset.aggregation_spec)

print("provenance:")
display(flow_dataset.provenance)

print("Último evento del FlowDataset:")
display(flow_dataset.metadata.get("events", [])[-1])

print("Descripción rápida de flow_value:")
display(flow_dataset.flows["flow_value"].describe())

Shape de flow_dataset.flows: (879, 9)
is_validated del FlowDataset: False


,flow_id,origin_h3_index,destination_h3_index,window_start_utc,window_end_utc,mode,user_gender,flow_count,flow_value
0,flow_0000000,85b2c50bfffffff,85b2c50bfffffff,2026-03-01,2026-03-02,bicycle,female,7,22.418
1,flow_0000001,85b2c50bfffffff,85b2c50bfffffff,2026-03-01,2026-03-02,bicycle,male,5,16.031
2,flow_0000002,85b2c50bfffffff,85b2c50bfffffff,2026-03-01,2026-03-02,bus,female,5,8.697
3,flow_0000003,85b2c50bfffffff,85b2c50bfffffff,2026-03-01,2026-03-02,car,female,7,21.281
4,flow_0000004,85b2c50bfffffff,85b2c50bfffffff,2026-03-01,2026-03-02,car,male,5,14.257
5,flow_0000005,85b2c50bfffffff,85b2c50bfffffff,2026-03-01,2026-03-02,car,other,5,11.354
6,flow_0000006,85b2c50bfffffff,85b2c50bfffffff,2026-03-01,2026-03-02,motorcycle,female,7,20.894
7,flow_0000007,85b2c50bfffffff,85b2c50bfffffff,2026-03-01,2026-03-02,other,male,5,11.219
8,flow_0000008,85b2c50bfffffff,85b2c50bfffffff,2026-03-01,2026-03-02,scooter,female,6,17.594
9,flow_0000009,85b2c50bfffffff,85b2c50bfffffff,2026-03-01,2026-03-02,scooter,male,8,24.550


,flow_id,origin_h3_index,destination_h3_index,mode,user_gender,window_start_utc,window_end_utc,flow_count,flow_value
0,flow_0000000,85b2c50bfffffff,85b2c50bfffffff,bicycle,female,2026-03-01,2026-03-02,7,22.418
1,flow_0000001,85b2c50bfffffff,85b2c50bfffffff,bicycle,male,2026-03-01,2026-03-02,5,16.031
2,flow_0000002,85b2c50bfffffff,85b2c50bfffffff,bus,female,2026-03-01,2026-03-02,5,8.697
3,flow_0000003,85b2c50bfffffff,85b2c50bfffffff,car,female,2026-03-01,2026-03-02,7,21.281
4,flow_0000004,85b2c50bfffffff,85b2c50bfffffff,car,male,2026-03-01,2026-03-02,5,14.257
5,flow_0000005,85b2c50bfffffff,85b2c50bfffffff,car,other,2026-03-01,2026-03-02,5,11.354
6,flow_0000006,85b2c50bfffffff,85b2c50bfffffff,motorcycle,female,2026-03-01,2026-03-02,7,20.894
7,flow_0000007,85b2c50bfffffff,85b2c50bfffffff,other,male,2026-03-01,2026-03-02,5,11.219
8,flow_0000008,85b2c50bfffffff,85b2c50bfffffff,scooter,female,2026-03-01,2026-03-02,6,17.594
9,flow_0000009,85b2c50bfffffff,85b2c50bfffffff,scooter,male,2026-03-01,2026-03-02,8,24.550


aggregation_spec:


{'h3_resolution': 5,
 'group_by': ['mode', 'user_gender'],
 'time_aggregation': 'day',
 'time_basis': 'origin',
 'min_trips_per_flow': 5,
 'keep_flow_to_trips': False,
 'require_validated': True,
 'strict': False,
 'max_issues': 200,
 'effective_flow_keys': ['origin_h3_index',
  'destination_h3_index',
  'window_start_utc',
  'window_end_utc',
  'mode',
  'user_gender']}

...

{'derived_from': [{'source_type': 'trips',
   'dataset_id': 'tripds_0ec20aea36a2479b958805ab2bdc46a0',
   'schema_version': '1.1',
   'n_rows': 18000}],
 'prior_events_summary': [{'op': 'import_trips',
   'ts_utc': '2026-04-06T07:12:03.425905+00:00',
   'summary': {'input_rows': 18000,
    'output_rows': 18000,
    'rows_dropped': 0,
    'n_fields_mapped': 24,
    'n_domain_mappings_applied': 5,
    'columns_added': ['origin_h3_index', 'destination_h3_index'],
    'columns_deleted': [],
    'domains_extended_count': 0,
    'temporal_tier': 'tier_1',
    'temporal_notes': None}},
  {'op': 'validate_trips',
   'ts_utc': '2026-04-06T07:14:05Z',
   'summary': {'ok': True,
    'n_rows': 18000,
    'n_issues': 0,
    'n_errors': 0,
    'n_warnings': 0,
    'n_info': 0,
    'counts_by_level': {'error': 0, 'warning': 0, 'info': 0},
    'counts_by_code': {},
    'checked_fields': ['activity_status',
     'day_type',
     'destination_h3_index',
     'destination_latitude',
     'destination_lon

Último evento del FlowDataset:


{'op': 'build_flows',
 'ts_utc': '2026-04-06T07:32:34.185795Z',
 'parameters': {'h3_resolution': 5,
  'group_by': ['mode', 'user_gender'],
  'time_aggregation': 'day',
  'time_basis': 'origin',
  'min_trips_per_flow': 5,
  'keep_flow_to_trips': False,
  'require_validated': True,
  'strict': False,
  'max_issues': 200},
 'summary': {'n_trips_in': 18000,
  'n_trips_eligible': 18000,
  'n_trips_dropped': 0,
  'n_flows_out': 879,
  'n_flow_to_trips_rows': None},
 'issues_summary': {'counts': {'info': 0, 'warning': 0, 'error': 0},
  'top_codes': []}}

Descripción rápida de flow_value:


count    879.000000
mean      24.433975
std       13.001795
min        7.377000
25%       15.186500
50%       20.291000
75%       29.995000
max       75.762000
Name: flow_value, dtype: float64

## 6. Exportación simple a layout flowmap

Ahora quiero materializar los flujos a un layout externo simple:
- `flows.csv`
- `locations.csv`
- `metadata.json`

Además voy a preservar algunas columnas extra en `flows.csv` para que el export también sea útil en Flowmap City.

In [32]:
export_options = ExportFlowsOptions(
    format="flowmap_blue",
    mode="overwrite",
    folder_name="demo_trip_to_flow_happy_path_v1",
    extra_flow_fields=[
        "mode",
        "user_gender",
        "window_start_utc",
        "window_end_utc",
        "flow_count",
    ],
)

export_result, export_report = export_flows(
    flow_dataset,
    output_root=str(EXPORT_ROOT),
    options=export_options,
)

assert export_report.ok is True

print("Export OK:", export_report.ok)
display(export_report.summary)
display(issues_to_df(export_report.issues))
print("Último evento del FlowDataset:")
display(flow_dataset.metadata.get("events", [])[-1])

Export OK: True


{'n_flows': 879,
 'n_locations': 8,
 'files_written': ['flows.csv', 'locations.csv', 'metadata.json']}

,level,code,message,field,source_field,row_count,details


Último evento del FlowDataset:


{'op': 'export_flows',
 'ts_utc': '2026-04-06T07:35:48.925699Z',
 'parameters': {'output_root': 'D:\\Memoria\\repos\\pylondrina\\data\\synthetic\\demo_outputs\\flow_exports',
  'export_dir': 'D:\\Memoria\\repos\\pylondrina\\data\\synthetic\\demo_outputs\\flow_exports\\demo_trip_to_flow_happy_path_v1',
  'format': 'flowmap_blue',
  'mode': 'overwrite',
  'folder_name': 'demo_trip_to_flow_happy_path_v1',
  'extra_flow_fields': ['mode',
   'user_gender',
   'window_start_utc',
   'window_end_utc',
   'flow_count']},
 'summary': {'n_flows': 879,
  'n_locations': 8,
  'files_written': ['flows.csv', 'locations.csv', 'metadata.json']},
 'issues_summary': {'counts': {'info': 0, 'warning': 0, 'error': 0},
  'top_codes': []}}

## 7. Inspección de los artefactos exportados

En esta etapa quiero revisar rápidamente que los tres artefactos hayan quedado bien:
- tabla de flows,
- tabla de locations,
- y sidecar de metadata.

In [33]:
flows_csv_path = Path(export_result.artifacts["flows"])
locations_csv_path = Path(export_result.artifacts["locations"])
metadata_json_path = Path(export_result.artifacts["metadata"])

flows_csv = pd.read_csv(flows_csv_path)
locations_csv = pd.read_csv(locations_csv_path)
metadata_json = json.loads(metadata_json_path.read_text(encoding="utf-8"))

print("export_dir:", export_result.export_dir)
print("artifacts:")
display(export_result.artifacts)

print("\nflows.csv")
display(flows_csv.head(10))

print("\nlocations.csv")
display(locations_csv.head(10))

print("\nmetadata.json -> export")
display(metadata_json["export"])

print("\nmetadata.json -> flow_dataset_ref (resumen)")
display(
    {
        "dataset_id": metadata_json["flow_dataset_ref"]["dataset_id"],
        "aggregation_spec": metadata_json["flow_dataset_ref"]["aggregation_spec"],
    }
)

export_dir: D:\Memoria\repos\pylondrina\data\synthetic\demo_outputs\flow_exports\demo_trip_to_flow_happy_path_v1
artifacts:


{'flows': 'D:\\Memoria\\repos\\pylondrina\\data\\synthetic\\demo_outputs\\flow_exports\\demo_trip_to_flow_happy_path_v1\\flows.csv',
 'locations': 'D:\\Memoria\\repos\\pylondrina\\data\\synthetic\\demo_outputs\\flow_exports\\demo_trip_to_flow_happy_path_v1\\locations.csv',
 'metadata': 'D:\\Memoria\\repos\\pylondrina\\data\\synthetic\\demo_outputs\\flow_exports\\demo_trip_to_flow_happy_path_v1\\metadata.json'}


flows.csv


,origin,dest,count,mode,user_gender,window_start_utc,window_end_utc,flow_count
0,85b2c50bfffffff,85b2c50bfffffff,22.418,bicycle,female,2026-03-01,2026-03-02,7
1,85b2c50bfffffff,85b2c50bfffffff,16.031,bicycle,male,2026-03-01,2026-03-02,5
2,85b2c50bfffffff,85b2c50bfffffff,8.697,bus,female,2026-03-01,2026-03-02,5
3,85b2c50bfffffff,85b2c50bfffffff,21.281,car,female,2026-03-01,2026-03-02,7
4,85b2c50bfffffff,85b2c50bfffffff,14.257,car,male,2026-03-01,2026-03-02,5
5,85b2c50bfffffff,85b2c50bfffffff,11.354,car,other,2026-03-01,2026-03-02,5
6,85b2c50bfffffff,85b2c50bfffffff,20.894,motorcycle,female,2026-03-01,2026-03-02,7
7,85b2c50bfffffff,85b2c50bfffffff,11.219,other,male,2026-03-01,2026-03-02,5
8,85b2c50bfffffff,85b2c50bfffffff,17.594,scooter,female,2026-03-01,2026-03-02,6
9,85b2c50bfffffff,85b2c50bfffffff,24.550,scooter,male,2026-03-01,2026-03-02,8



locations.csv


,id,name,lat,lon
0,85b2c50bfffffff,85b2c50bfffffff,-33.496369,-70.530593
1,85b2c51bfffffff,85b2c51bfffffff,-33.356983,-70.517466
2,85b2c543fffffff,85b2c543fffffff,-33.501549,-70.835463
3,85b2c547fffffff,85b2c547fffffff,-33.568667,-70.689561
4,85b2c553fffffff,85b2c553fffffff,-33.362051,-70.822135
5,85b2c557fffffff,85b2c557fffffff,-33.429365,-70.676326
6,85b2c573fffffff,85b2c573fffffff,-33.635475,-70.543737
7,85b2c5cffffffff,85b2c5cffffffff,-33.289783,-70.663106



metadata.json -> export


{'parameters': {'output_root': 'D:\\Memoria\\repos\\pylondrina\\data\\synthetic\\demo_outputs\\flow_exports',
  'export_dir': 'D:\\Memoria\\repos\\pylondrina\\data\\synthetic\\demo_outputs\\flow_exports\\demo_trip_to_flow_happy_path_v1',
  'format': 'flowmap_blue',
  'mode': 'overwrite',
  'folder_name': 'demo_trip_to_flow_happy_path_v1',
  'extra_flow_fields': ['mode',
   'user_gender',
   'window_start_utc',
   'window_end_utc',
   'flow_count']},
 'summary': {'n_flows': 879,
  'n_locations': 8,
  'files_written': ['flows.csv', 'locations.csv', 'metadata.json']},
 'count_source': 'flow_value'}


metadata.json -> flow_dataset_ref (resumen)


{'dataset_id': 'flows_4d5d92f55041',
 'aggregation_spec': {'h3_resolution': 5,
  'group_by': ['mode', 'user_gender'],
  'time_aggregation': 'day',
  'time_basis': 'origin',
  'min_trips_per_flow': 5,
  'keep_flow_to_trips': False,
  'require_validated': True,
  'strict': False,
  'max_issues': 200,
  'effective_flow_keys': ['origin_h3_index',
   'destination_h3_index',
   'window_start_utc',
   'window_end_utc',
   'mode',
   'user_gender']}}

In [35]:
print("Resumen final")
display(
    {
        "raw_shape": raw_df.shape,
        "trips_shape": trips.data.shape,
        "flow_shape": flow_dataset.flows.shape,
        "import_ok": import_report.ok,
        "validation_ok": validation_report.ok,
        "build_ok": flow_build_report.ok,
        "export_ok": export_report.ok,
        "export_dir": export_result.export_dir,
    }
)

Resumen final


{'raw_shape': (18000, 40),
 'trips_shape': (18000, 42),
 'flow_shape': (879, 9),
 'import_ok': True,
 'validation_ok': True,
 'build_ok': True,
 'export_ok': True,
 'export_dir': 'D:\\Memoria\\repos\\pylondrina\\data\\synthetic\\demo_outputs\\flow_exports\\demo_trip_to_flow_happy_path_v1'}